In [1]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import random
import json
import os

from utils.phase import Phase
from utils.model_utils import criterion_to_str
from utils.model_utils import compare_results_strata
from data_augmentation.monte_carlo_loader import MonteCarloLoader
from data_augmentation.datasets.dense_monte_carlo_dataset import DenseMonteCarloDataset
from model.architecture.dense_monte_carlo import DenseMonteCarloNet
from model.loss_function.dense_monte_carlo.custom_dense_MC_loss import CustomDenseMCLoss
from model.model_loader import ModelLoader
from model.trainers.dense_monte_carlo_trainer import DenseMonteCarloTrainer
from model.trainers.trainer import save_state_model, load_model, get_hyperparameter_model_str
from analytic_methods.EVM_method import EVM
from visualization.results.metrics_visualizator import MetricsVisualizator
from visualization.data.weights_visualizator import WeightsVisualizator
from metrics.evaluator import Evaluator

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device used: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Device used: cuda
GPU: NVIDIA GeForce RTX 2060


In [2]:
def build_and_train_model_monte_carlo(
    neurons=32,
    layers=2,
    expansion_factor=4,
    dropout_rate=0.3,
    negative_slope=0.1,
    learning_rate=0.001,
    batch_size=1024,
    batch_norm=True,
    epochs=500,
    n_criteria=5,
    es_patience=15,
    reduce_lr_on_plateau=True,
    rlr_patience=5,
    train_dataset=None,
    valid_dataset=None,
    criterion=None,
    device=None,
):
    target_device = device if device is not None else globals().get('device', torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
    target_n_criteria = globals().get('CRITERIA_NUM', n_criteria)

    if train_dataset is None:
        train_dataset = globals().get('train_dataset')
    if valid_dataset is None:
        valid_dataset = globals().get('valid_dataset')

    if train_dataset is None:
        train_dataset = DenseMonteCarloDataset(
            num_criteria=target_n_criteria,
            phase=Phase.TRAIN,
            augment=True,
        )
        globals()['train_dataset'] = train_dataset

    if valid_dataset is None:
        valid_dataset = DenseMonteCarloDataset(
            num_criteria=target_n_criteria,
            phase=Phase.VALID,
            augment=False,
        )
        globals()['valid_dataset'] = valid_dataset

    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    valid_dataloader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)

    model = DenseMonteCarloNet(
        n_criteria=target_n_criteria,
        neurons=neurons,
        layers=layers,
        expansion_factor=expansion_factor,
        dropout_rate=dropout_rate,
        negative_slope=negative_slope,
        use_batch_norm=batch_norm,
    ).to(target_device)

    if criterion is None:
        criterion = globals().get('criterion')
        if criterion is None:
            cop = globals().get('COP', 0.5)
            fit = globals().get('FIT', 0.45)
            stab = globals().get('STAB', 0.05)
            p = globals().get('P', 2)
            margin = globals().get('MARGIN', 0.05)
            norm_loss = globals().get('NORMALIZE_INITIAL_LOSS', False)
            cop_mode = globals().get('COP_MODE', 'margin')
            stab_mode = globals().get('STAB_MODE', 'kullback')
            criterion = CustomDenseMCLoss(
                n_criteria=target_n_criteria,
                lambda_cop=cop,
                lambda_fit=fit,
                lambda_stab=stab,
                margin=margin,
                p=p,
                cop_mode=cop_mode,
                stab_mode=stab_mode,
                normalize_initial_loss=norm_loss,
            ).to(target_device)

    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    trainer = DenseMonteCarloTrainer(model, optimizer, criterion, target_device, model_type="dense_optimization")

    target_epochs = globals().get('EPOCHS', epochs)

    model, history = trainer.train(
        train_dataloader=train_dataloader,
        valid_dataloader=valid_dataloader,
        epochs=target_epochs,
        early_stopping=True,
        es_patience=es_patience,
        reduce_lr_on_plateau=reduce_lr_on_plateau,
        rlr_patience=rlr_patience,
    )

    val_loss = history.get(
        'best_valid_loss',
        min(history['valid_loss']) if history.get('valid_loss') else float('inf'),
    )

    params = {
        'neurons': neurons,
        'layers': layers,
        'expansion_factor': expansion_factor,
        'dropout_rate': dropout_rate,
        'negative_slope': negative_slope,
        'learning_rate': learning_rate,
        'batch_size': batch_size,
        'batch_norm': batch_norm,
    }
    history['params'] = params
    save_state_model(model, history, params=params)

    return trainer, val_loss, history


In [ ]:
def optimize_hyperparameter(
    param_name: str,
    param_values: list,
    current_best_params: dict,
    X_val: np.ndarray | torch.Tensor | None = None,
    y_val: np.ndarray | torch.Tensor | None = None,
    metric: str = "mae",
) -> dict:
    print(f"\n{'='*40}")
    print(f" Parameter optimization: {param_name.upper()}")
    print(f"{'='*40}")

    if X_val is None or y_val is None:
        valid_dataset = globals().get("valid_dataset")
        if valid_dataset is not None:
            if hasattr(valid_dataset, "matrices") and hasattr(valid_dataset, "targets"):
                X_val = valid_dataset.matrices.cpu().numpy()
                y_val = valid_dataset.targets.cpu().numpy()

    if isinstance(X_val, torch.Tensor):
        X_val = X_val.cpu().numpy()
    if isinstance(y_val, torch.Tensor):
        y_val = y_val.cpu().numpy()

    evaluator = Evaluator()
    results = []

    for value in param_values:
        params_to_pass = current_best_params.copy()
        params_to_pass[param_name] = value

        print(f"\nTraining for {param_name} = {value}")

        trainer, val_loss, history = build_and_train_model_monte_carlo(**params_to_pass)

        if X_val is not None and y_val is not None:
            y_pred = trainer.predict_batch(X_val)
            if y_val.ndim != y_pred.ndim:
                y_val_eval = np.squeeze(y_val)
                y_pred_eval = np.squeeze(y_pred)
            else:
                y_val_eval = y_val
                y_pred_eval = y_pred

            mae_scores, tau_scores, top1_scores, spearman_scores = evaluator.evaluate_batch(y_val_eval, y_pred_eval)
            val_mae = float(np.mean(mae_scores))
            val_tau = float(np.mean(tau_scores))
        else:
            hist_mae = history.get("valid_mae", [])
            val_mae = min(hist_mae) if len(hist_mae) > 0 else val_loss
            val_tau = 0.0

        print(
            f"-> Validation MAE: {val_mae:.4f}, Tau: {val_tau:.4f}, "
            f"Loss: {val_loss:.4f} for {param_name}={value}"
        )

        results.append((value, val_mae, val_loss, val_tau))

    if metric == "mae":
        best_item = min(results, key=lambda x: x[1])
    elif metric == "loss":
        best_item = min(results, key=lambda x: x[2])
    elif metric == "tau":
        best_item = max(results, key=lambda x: x[3])
    elif metric == "top1":
        best_item = max(results, key=lambda x: x[4])
    else:
        best_item = min(results, key=lambda x: x[1])

    best_value, best_mae, best_loss, best_tau = best_item
    current_best_params[param_name] = best_value

    print(
        f"\nThe best {param_name}: {best_value} "
        f"(val_mae: {best_mae:.4f}, val_loss: {best_loss:.4f}, "
        f"tau: {best_tau:.4f})"
    )
    print(f"Temporary best hyperparameters: {current_best_params}")

    return current_best_params


In [4]:
best_params = {"neurons": 128, "layers": 4, "expansion_factor": 4 , "dropout_rate": 0.3,
               "negative_slope": 0.1, "learning_rate": 0.001, "batch_size": 256, "batch_norm": False}

params = {
    "neurons" : [4, 8, 16, 32, 64, 128, 256, 512],
    "layers" : [3, 4, 5, 6, 7, 8],
    "expansion_factor" : [2, 4, 6, 8],
    "dropout_rate" : [0, 0.1, 0.2, 0.3, 0.4, 0.5],
    "negative_slope" : [0.001, 0.01, 0.1, 0.5],
    "learning_rate" : [0.0001, 0.001, 0.01],
    "batch_size" : [64, 128, 256, 512, 1024, 2048],
    "batch_norm" : [True, False],
}

In [ ]:
CRITERIA_NUM = globals().get("CRITERIA_NUM", 5)

if "valid_dataset" not in globals() or globals().get("valid_dataset") is None:
    valid_dataset = DenseMonteCarloDataset(num_criteria=CRITERIA_NUM, phase=Phase.VALIDATION, augment=False)
X_val = valid_dataset.matrices.cpu().numpy()
y_val = valid_dataset.targets.cpu().numpy()

if "test_dataset" not in globals() or globals().get("test_dataset") is None:
    test_dataset = DenseMonteCarloDataset(num_criteria=CRITERIA_NUM, phase=Phase.TEST, augment=False)
X_test = test_dataset.matrices.cpu().numpy()
y_test = test_dataset.targets.cpu().numpy()

num_orders_to_try = 5
order_results = []
initial_best_params = best_params.copy()
evaluator = Evaluator()

for i in range(num_orders_to_try):
    params_list = list(params.items())
    random.shuffle(params_list)
    shuffled_params = dict(params_list)
    order_names = list(shuffled_params.keys())

    current_best_params = initial_best_params.copy()

    print(f"\n--- Order {i+1}: {order_names} ---")
    for key, values in shuffled_params.items():
        current_best_params = optimize_hyperparameter(key, values, current_best_params, X_val, y_val)

    final_trainer, final_val_loss, final_history = build_and_train_model_monte_carlo(**current_best_params)

    y_pred_test = final_trainer.predict_batch(X_test)
    if y_test.ndim != y_pred_test.ndim:
        y_test_eval = np.squeeze(y_test)
        y_pred_eval = np.squeeze(y_pred_test)
    else:
        y_test_eval = y_test
        y_pred_eval = y_pred_test

    mae_scores, tau_scores, top1_scores, spearman_scores = evaluator.evaluate_batch(y_test_eval, y_pred_eval)
    test_mae = float(np.mean(mae_scores))
    test_tau = float(np.mean(tau_scores))

    print(f"Final test MAE for order {i+1}: {test_mae:.4f} (Tau: {test_tau:.4f})")
    order_results.append((order_names, current_best_params.copy(), test_mae, test_tau))

best_order_names, best_hyperparams, final_test_mae, final_test_tau = min(
    order_results, key=lambda x: x[2]
)

print("\nFinal Best Hyperparameters")
print(f"The best set of hyperparameters was obtained using the following order of optimization: {best_order_names}")
print("\nBest Hyperparameters:")
for key, value in best_hyperparams.items():
    print(f"  - {key}: {value}")
print(f"\nFinal Test MAE: {final_test_mae:.4f}, Kendall Tau: {final_test_tau:.4f}")


--- Order 1: ['learning_rate', 'expansion_factor', 'negative_slope', 'neurons', 'batch_norm', 'batch_size', 'layers', 'dropout_rate'] ---

 Parameter optimization: LEARNING_RATE

Training for learning_rate = 0.0001
Epoch number [10/500] | Train Loss: 0.285406 | Valid Loss: 0.195370 | Valid MAE: 0.0446
Epoch number [20/500] | Train Loss: 0.266432 | Valid Loss: 0.183632 | Valid MAE: 0.0435
Epoch number [30/500] | Train Loss: 0.259637 | Valid Loss: 0.180828 | Valid MAE: 0.0433
Epoch number [40/500] | Train Loss: 0.256606 | Valid Loss: 0.180097 | Valid MAE: 0.0432
Epoch number [50/500] | Train Loss: 0.255011 | Valid Loss: 0.179941 | Valid MAE: 0.0431
Epoch number [60/500] | Train Loss: 0.253808 | Valid Loss: 0.178369 | Valid MAE: 0.0430
Epoch number 62: Learning Rate was reduced to 0.000050
Epoch number [70/500] | Train Loss: 0.253059 | Valid Loss: 0.177938 | Valid MAE: 0.0429
Epoch number [80/500] | Train Loss: 0.252318 | Valid Loss: 0.177451 | Valid MAE: 0.0429
Epoch number 89: Learning

ValueError: not enough values to unpack (expected 5, got 4)